In [ ]:
import tkinter as tk
from tkinter import ttk
import random
import math
from collections import deque
import heapq

TRANG_THAI_BAN_DAU = [
    [1, 2, 3],
    [4, 5, 6],
    [0, 7, 8]
]

KET_QUA_DICH = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

def lam_phang_ma_tran(ma_tran):
    mảng_phẳng = []
    for hang in ma_tran:
        mảng_phẳng.extend(hang)
    return mảng_phẳng

TRANG_THAI_DICH_PHANG = tuple(lam_phang_ma_tran(KET_QUA_DICH))
TRANG_THAI_DAU_PHANG = lam_phang_ma_tran(TRANG_THAI_BAN_DAU)

CAC_HUONG = ["Trai", "Phai", "Len", "Xuong"]
HUONG_NGUOC = {"Trai": "Phai", "Phai": "Trai", "Len": "Xuong", "Xuong": "Len"}

class GiaoDienTongHopTatCa:
    def __init__(self, cua_so):
        self.cua_so = cua_so
        self.cua_so.title("8-Puzzle - Ultimate Algorithms Simulator")

        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.dang_tu_dong = False
        self._after_id = None

        khung_tren = tk.Frame(cua_so)
        khung_tren.pack(pady=10, fill="x")

        self.nhan_trang_thai = tk.Label(
            khung_tren, text="Số bước: 0", anchor="w", font=("Arial", 11)
        )
        self.nhan_trang_thai.pack(side="left", padx=10, fill="x", expand=True)

        self.bien_thuat_toan = tk.StringVar(value="A* Search")
        danh_sach_thuat_toan = [
            "BFS", "DFS", "UCS", "IDS", 
            "A* Search", "IDA*", "Greedy", 
            "Simple Hill Climbing", "Steepest Ascent Hill", 
            "Stochastic Hill", "Random Restart Hill", 
            "Simulated Annealing", "Local Beam", 
            "Belief Search", "Belief Goal",
            "Backtracking", "And-Or Search", "Forward Checking", "AC3"
        ]
        
        menu_thuat_toan = ttk.Combobox(
            khung_tren, textvariable=self.bien_thuat_toan, 
            values=danh_sach_thuat_toan, state="readonly", width=20
        )
        menu_thuat_toan.pack(side="right", padx=5)

        tk.Button(khung_tren, text="Chạy Thuật Toán", command=self.giai_thuat_toan, bg="#ffe4e1", font=("Arial", 10, "bold")).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Dừng", command=self.dung_tu_dong).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Trộn", command=self.tron).pack(side="right", padx=5)
        tk.Button(khung_tren, text="Đặt Lại", command=self.dat_lai).pack(side="right", padx=5)

        khung_chinh = tk.Frame(cua_so)
        khung_chinh.pack(padx=10, pady=10, fill="both", expand=True)

        khung_trai = tk.Frame(khung_chinh)
        khung_trai.pack(side="left", fill="y")

        khung_ban_co = tk.Frame(khung_trai, bg="darkgoldenrod2", padx=8, pady=8)
        khung_ban_co.pack(anchor="n")

        self.danh_sach_o = []
        for hang in range(3):
            for cot in range(3):
                o = tk.Label(
                    khung_ban_co, text="", width=4, height=2,
                    font=("Arial", 22, "bold"), bg="white", relief="raised", borderwidth=3
                )
                o.grid(row=hang, column=cot, padx=4, pady=4)
                self.danh_sach_o.append(o)

        khung_dich = tk.Frame(khung_trai, bg="#888", padx=10, pady=10)
        khung_dich.pack(pady=20, anchor="n")
        tk.Label(khung_dich, text="GOAL STATE", font=("Arial", 12, "bold"), bg="#888", fg="white").pack(pady=5)

        luoi_dich = tk.Frame(khung_dich, bg="#888")
        luoi_dich.pack()
        for i, gia_tri in enumerate(TRANG_THAI_DICH_PHANG):
            hang, cot = divmod(i, 3)
            chu = "" if gia_tri == 0 else str(gia_tri)
            mau_nen = "#bbb" if gia_tri != 0 else "#999"
            o_dich = tk.Label(luoi_dich, text=chu, width=4, height=2, font=("Arial", 16, "bold"), bg=mau_nen, relief="ridge", borderwidth=2)
            o_dich.grid(row=hang, column=cot, padx=3, pady=3)

        khung_phai = tk.Frame(khung_chinh, padx=10)
        khung_phai.pack(side="left", fill="both", expand=True)
        tk.Label(khung_phai, text="📜 Lịch Sử Bước Đi", font=("Arial", 11, "bold")).pack(anchor="w")

        scrollbar = tk.Scrollbar(khung_phai)
        scrollbar.pack(side="right", fill="y")

        self.hop_lich_su = tk.Text(
            khung_phai, width=25, height=25, font=("Courier", 11),
            yscrollcommand=scrollbar.set, state="disabled", bg="#f8f9fa"
        )
        self.hop_lich_su.pack(side="left", fill="both", expand=True)
        scrollbar.config(command=self.hop_lich_su.yview)

        self.ve_giao_dien()

    def ve_giao_dien(self):
        for i, gia_tri in enumerate(self.ban_co):
            o = self.danh_sach_o[i]
            if gia_tri == 0: o.config(text="", bg="#555", relief="sunken")
            else: o.config(text=str(gia_tri), bg="#f2f2f2", relief="raised")
            
    def dat_lai(self):
        self.dung_tu_dong()
        self.ban_co = list(TRANG_THAI_DAU_PHANG)
        self.so_buoc = 0
        self.ve_giao_dien()
        self.xoa_lich_su()
        self.nhan_trang_thai.config(text="Đã đặt lại bàn cờ.", fg="black")

    def xoa_lich_su(self):
        self.hop_lich_su.config(state="normal")
        self.hop_lich_su.delete(1.0, tk.END)
        self.hop_lich_su.config(state="disabled")

    def in_trang_thai_ra_lich_su(self, trang_thai):
        for i in range(3):
            hang = trang_thai[i*3:(i+1)*3]
            hang_str = "   ".join(str(x) if x != 0 else "_" for x in hang)
            self.hop_lich_su.insert(tk.END, f"  {hang_str}\n")
        self.hop_lich_su.insert(tk.END, "----------------\n")

    def xuat_lich_su_ra_man_hinh(self, duong_di):
        self.hop_lich_su.config(state="normal")
        self.hop_lich_su.delete(1.0, tk.END)
        
        trang_thai_hien_tai = list(self.ban_co)
        self.hop_lich_su.insert(tk.END, "TRẠNG THÁI BẮT ĐẦU:\n")
        self.in_trang_thai_ra_lich_su(trang_thai_hien_tai)

        for i, buoc in enumerate(duong_di, 1):
            self.hop_lich_su.insert(tk.END, f"Bước {i}: Đi {buoc}\n")
            
            vi_tri_trong = trang_thai_hien_tai.index(0)
            hang, cot = divmod(vi_tri_trong, 3)
            if buoc == "Trai": h_moi, c_moi = hang, cot - 1
            elif buoc == "Phai": h_moi, c_moi = hang, cot + 1
            elif buoc == "Len": h_moi, c_moi = hang - 1, cot
            elif buoc == "Xuong": h_moi, c_moi = hang + 1, cot
            
            vi_tri_moi = h_moi * 3 + c_moi
            trang_thai_hien_tai[vi_tri_trong], trang_thai_hien_tai[vi_tri_moi] = trang_thai_hien_tai[vi_tri_moi], trang_thai_hien_tai[vi_tri_trong]
            
            self.in_trang_thai_ra_lich_su(trang_thai_hien_tai)

        self.hop_lich_su.config(state="disabled")

    def di_chuyen_o_trong(self, huong):
        vi_tri_trong = self.ban_co.index(0)
        hang, cot = divmod(vi_tri_trong, 3)
        if huong == "Trai": h_moi, c_moi = hang, cot - 1
        elif huong == "Phai": h_moi, c_moi = hang, cot + 1
        elif huong == "Len": h_moi, c_moi = hang - 1, cot
        elif huong == "Xuong": h_moi, c_moi = hang + 1, cot
        else: return False
        if not (0 <= h_moi < 3 and 0 <= c_moi < 3): return False
        v_moi = h_moi * 3 + c_moi
        self.ban_co[vi_tri_trong], self.ban_co[v_moi] = self.ban_co[v_moi], self.ban_co[vi_tri_trong]
        return True

    def tron(self, so_buoc=30):
        self.dung_tu_dong()
        self.xoa_lich_su()
        self.ban_co = list(TRANG_THAI_DICH_PHANG)
        h_truoc = None
        for _ in range(so_buoc):
            cac_huong = ["Trai", "Phai", "Len", "Xuong"]
            random.shuffle(cac_huong)
            for h in cac_huong:
                if h_truoc and h == HUONG_NGUOC[h_truoc]: continue
                if self.di_chuyen_o_trong(h):
                    h_truoc = h
                    break
        self.ve_giao_dien()
        self.nhan_trang_thai.config(text="Đã trộn ngẫu nhiên.", fg="black")

    def _hang_xom(self, trang_thai):
        idx0 = trang_thai.index(0)
        r, c = divmod(idx0, 3)
        kq = []
        for h in CAC_HUONG:
            if h == "Trai": r2, c2 = r, c - 1
            elif h == "Phai": r2, c2 = r, c + 1
            elif h == "Len": r2, c2 = r - 1, c
            else: r2, c2 = r + 1, c
            if 0 <= r2 < 3 and 0 <= c2 < 3:
                lst = list(trang_thai)
                lst[idx0], lst[r2*3+c2] = lst[r2*3+c2], lst[idx0]
                kq.append((tuple(lst), h))
        return kq

    def _manhattan(self, trang_thai):
        dist = 0
        for i, val in enumerate(trang_thai):
            if val != 0:
                g_idx = TRANG_THAI_DICH_PHANG.index(val)
                dist += abs(i//3 - g_idx//3) + abs(i%3 - g_idx%3)
        return dist

    def conflict(self, trang_thai):
        dem = 0
        for i in range(9):
            if trang_thai[i] != 0 and trang_thai[i] != TRANG_THAI_DICH_PHANG[i]:
                dem += 1
        return dem

    def bfs(self, start):
        q = deque([(start, [])])
        visited = set([start])
        while q:
            node, path = q.popleft()
            if node == TRANG_THAI_DICH_PHANG: return path
            for nxt, act in self._hang_xom(node):
                if nxt not in visited:
                    visited.add(nxt)
                    q.append((nxt, path + [act]))
        return None

    def dfs(self, start):
        stack = [(start, [])]
        visited = {start: 0}
        while stack:
            node, path = stack.pop()
            if node == TRANG_THAI_DICH_PHANG: return path
            if len(path) > 40: continue
            for nxt, act in self._hang_xom(node):
                if nxt not in visited or len(path)+1 < visited[nxt]:
                    visited[nxt] = len(path)+1
                    stack.append((nxt, path + [act]))
        return None

    def ucs(self, start):
        dem = 0
        pq = [(0, dem, start, [])] 
        visited = {start: 0}
        while pq:
            g, _, node, path = heapq.heappop(pq)
            if node == TRANG_THAI_DICH_PHANG: return path
            for nxt, act in self._hang_xom(node):
                if nxt not in visited or g + 1 < visited[nxt]:
                    visited[nxt] = g + 1
                    dem += 1
                    heapq.heappush(pq, (g + 1, dem, nxt, path + [act]))
        return None

    def ids(self, start):
        def dls(node, path, limit):
            if node == TRANG_THAI_DICH_PHANG: return path
            if limit == 0: return None
            for nxt, act in self._hang_xom(node):
                res = dls(nxt, path + [act], limit - 1)
                if res: return res
            return None
        for i in range(50): 
            res = dls(start, [], i)
            if res is not None: return res
        return None

    def greedy(self, start):
        dem = 0
        pq = [(self._manhattan(start), dem, start, [])]
        visited = set([start])
        while pq:
            _, _, node, path = heapq.heappop(pq)
            if node == TRANG_THAI_DICH_PHANG: return path
            for nxt, act in self._hang_xom(node):
                if nxt not in visited:
                    visited.add(nxt)
                    dem += 1
                    heapq.heappush(pq, (self._manhattan(nxt), dem, nxt, path + [act]))
        return None

    def astar(self, start):
        dem = 0
        pq = [(self._manhattan(start), dem, start, [])]
        visited = {start: 0}
        while pq:
            _, _, node, path = heapq.heappop(pq)
            if node == TRANG_THAI_DICH_PHANG: return path
            g = len(path) + 1
            for nxt, act in self._hang_xom(node):
                if nxt not in visited or g < visited[nxt]:
                    visited[nxt] = g
                    f = g + self._manhattan(nxt)
                    dem += 1
                    heapq.heappush(pq, (f, dem, nxt, path + [act]))
        return None

    def ida_star(self, bat_dau):
        def search(duong_di, cac_buoc, g, nguong):
            node = duong_di[-1]
            f = g + self._manhattan(node)
            if f > nguong: return f
            if node == TRANG_THAI_DICH_PHANG: return "OK"
            min_v = float('inf')
            for nxt, h in self._hang_xom(node):
                if nxt not in duong_di:
                    duong_di.append(nxt)
                    cac_buoc.append(h)
                    res = search(duong_di, cac_buoc, g + 1, nguong)
                    if res == "OK": return "OK"
                    if res < min_v: min_v = res
                    duong_di.pop()
                    cac_buoc.pop()
            return min_v
        nguong = self._manhattan(bat_dau)
        duong_di, cac_buoc = [bat_dau], []
        while True:
            res = search(duong_di, cac_buoc, 0, nguong)
            if res == "OK": return cac_buoc
            if res == float('inf'): return None
            nguong = res

    def simple_hill(self, start):
        current = start
        path = []
        for _ in range(1000): 
            if current == TRANG_THAI_DICH_PHANG: return path
            h_cur = self._manhattan(current)
            found = False
            for nxt, act in self._hang_xom(current):
                if self._manhattan(nxt) < h_cur:
                    current = nxt
                    path.append(act)
                    found = True
                    break
            if not found: return None 
        return None

    def steepest_ascent(self, start):
        current = start
        path = []
        for _ in range(1000):
            if current == TRANG_THAI_DICH_PHANG: return path
            h_cur = self._manhattan(current)
            neighbors = self._hang_xom(current)
            best_nxt, best_act = min(neighbors, key=lambda x: self._manhattan(x[0]))
            if self._manhattan(best_nxt) >= h_cur: return None 
            current = best_nxt
            path.append(best_act)
        return None

    def stochastic_hill(self, start):
        current = start
        path = []
        for _ in range(1000):
            if current == TRANG_THAI_DICH_PHANG: return path
            h_cur = self._manhattan(current)
            better_neighbors = [x for x in self._hang_xom(current) if self._manhattan(x[0]) < h_cur]
            if not better_neighbors: return None
            current, act = random.choice(better_neighbors)
            path.append(act)
        return None

    def random_restart_hill(self, start):
        for _ in range(10): 
            current = start if _ == 0 else tuple(random.sample(range(9), 9)) 
            path = []
            while True:
                if current == TRANG_THAI_DICH_PHANG: return path
                neighbors = self._hang_xom(current)
                best_nxt, best_act = min(neighbors, key=lambda x: self._manhattan(x[0]))
                if self._manhattan(best_nxt) >= self._manhattan(current): break 
                current = best_nxt
                path.append(best_act)
        return None

    def simulated_annealing(self, start):
        current = start
        path = []
        T = 100.0
        for _ in range(5000):
            if current == TRANG_THAI_DICH_PHANG: return path
            neighbors = self._hang_xom(current)
            nxt, act = random.choice(neighbors)
            delta = self._manhattan(current) - self._manhattan(nxt) 
            if delta > 0 or random.random() < math.exp(delta / T):
                current = nxt
                path.append(act)
            T *= 0.99
            if T < 0.01: break
        return None

    def local_beam(self, start, k=3):
        beam = [(self._manhattan(start), start, [])]
        for _ in range(500):
            next_states = []
            for _, node, path in beam:
                if node == TRANG_THAI_DICH_PHANG: return path
                for nxt, act in self._hang_xom(node):
                    next_states.append((self._manhattan(nxt), nxt, path + [act]))
            next_states.sort(key=lambda x: x[0])
            beam = next_states[:k]
            if not beam: break
        return None

    def get_belief_successors(self, belief_state, action):
        """
        Hàm bổ trợ: Dịch chuyển ô trống (0) đồng loạt trên TẤT CẢ các cấu trúc 
        bàn cờ giả định có trong tập Belief hiện tại theo cùng một hướng 'action'.
        """
        new_belief = set()
        for state in belief_state:
            idx = state.index(0)
            r, c = idx // 3, idx % 3
            
            new_r, new_c = r, c
            if action == "Trai" and c > 0: new_c -= 1
            elif action == "Phai" and c < 2: new_c += 1
            elif action == "Len" and r > 0: new_r -= 1
            elif action == "Xuong" and r < 2: new_r += 1
            else:
                new_belief.add(state)
                continue
                
            new_idx = new_r * 3 + new_c
            lst = list(state)
            lst[idx], lst[new_idx] = lst[new_idx], lst[idx]
            new_belief.add(tuple(lst))
            
        return tuple(sorted(list(new_belief)))

    def belief_search(self, start_state):
        """
        Thuật toán Belief Search sử dụng BFS để tìm kiếm trên không gian niềm tin.
        Cam kết trả ra kết quả trực tiếp cho luồng giao diện UI xử lý.
        """
       
        lst_gia_dinh = list(start_state)
        idx0 = lst_gia_dinh.index(0)
        r, c = idx0 // 3, idx0 % 3
        target_idx = idx0
        if c > 0: target_idx = idx0 - 1
        elif c < 2: target_idx = idx0 + 1
        elif r > 0: target_idx = idx0 - 3
        elif r < 2: target_idx = idx0 + 3
        
        lst_gia_dinh[idx0], lst_gia_dinh[target_idx] = lst_gia_dinh[target_idx], lst_gia_dinh[idx0]
        state_gia_dinh = tuple(lst_gia_dinh)
        initial_belief = tuple(sorted(list(set([start_state, state_gia_dinh]))))
        
        queue = deque([(initial_belief, [])])
        visited = {initial_belief}
        max_nodes = 8000 
        nodes_count = 0

        while queue:
            current_belief, path = queue.popleft()
            nodes_count += 1
            if all(s == TRANG_THAI_DICH_PHANG for s in current_belief):
                return path
                
            if nodes_count > max_nodes:
                return None 
                
            for action in ["Trai", "Phai", "Len", "Xuong"]:
                next_belief = self.get_belief_successors(current_belief, action)
                
                if next_belief not in visited:
                    visited.add(next_belief)
                    queue.append((next_belief, path + [action]))
                    
        return None

    def belief_goal(self, start_state):
        """
        Thuật toán Belief Goal sử dụng Min-Heap (Priority Queue) để tối ưu hóa,
        ưu tiên chọn các đường đi làm giảm số lượng trạng thái không chắc chắn.
        """
        lst_gia_dinh = list(start_state)
        idx0 = lst_gia_dinh.index(0)
        r, c = idx0 // 3, idx0 % 3
        
        target_idx = idx0
        if c > 0: target_idx = idx0 - 1
        elif c < 2: target_idx = idx0 + 1
        elif r > 0: target_idx = idx0 - 3
        elif r < 2: target_idx = idx0 + 3
        
        lst_gia_dinh[idx0], lst_gia_dinh[target_idx] = lst_gia_dinh[target_idx], lst_gia_dinh[idx0]
        state_gia_dinh = tuple(lst_gia_dinh)

        initial_belief = tuple(sorted(list(set([start_state, state_gia_dinh]))))
        priority_queue = []
        unique_counter = 0
        heapq.heappush(priority_queue, (len(initial_belief), unique_counter, initial_belief, []))
        
        visited = {initial_belief}
        max_nodes = 8000
        nodes_count = 0

        while priority_queue:
            _, _, current_belief, path = heapq.heappop(priority_queue)
            nodes_count += 1
            
            if all(s == TRANG_THAI_DICH_PHANG for s in current_belief):
                return path
                
            if nodes_count > max_nodes:
                return None
                
            for action in ["Trai", "Phai", "Len", "Xuong"]:
                next_belief = self.get_belief_successors(current_belief, action)
                
                if next_belief not in visited:
                    visited.add(next_belief)
                    unique_counter += 1
                    f_score = len(path) + 1 + len(next_belief)
                    heapq.heappush(priority_queue, (f_score, unique_counter, next_belief, path + [action]))
                    
        return None

    def backtracking(self, start):
        visited = set()
        def backtrack(node, path, depth_limit=15):
            if node == TRANG_THAI_DICH_PHANG:
                return path
            if len(path) >= depth_limit:
                return None
            visited.add(node)
            for nxt, act in self._hang_xom(node):
                if nxt not in visited:
                    res = backtrack(nxt, path + [act], depth_limit)
                    if res is not None:
                        return res
            visited.remove(node)
            return None
        return backtrack(start, [])

    def and_or_search(self, start):
        def or_search(state, visited, depth=0):
            if state == TRANG_THAI_DICH_PHANG:
                return []
            if state in visited or depth > 15:
                return None
            visited.add(state)
            for nxt, act in self._hang_xom(state):
                plan = and_search([nxt], visited, depth + 1)
                if plan is not None:
                    visited.remove(state)
                    return [act] + plan
            visited.remove(state)
            return None

        def and_search(states, visited, depth):
            for s in states:
                p = or_search(s, visited, depth)
                if p is None: return None
                return p
            return []

        visited = set()
        return or_search(start, visited, 0)

    def forward_checking_search(self, start):
        visited = set()
        def backtrack_fc(node, path, depth_limit=20):
            if node == TRANG_THAI_DICH_PHANG:
                return path
            if len(path) >= depth_limit:
                return None
            visited.add(node)
            
            valid_neighbors = []
            for nxt, act in self._hang_xom(node):
                if nxt in visited: continue
                nxt_neighbors = self._hang_xom(nxt)
                has_valid_forward = False
                for nnxt, _ in nxt_neighbors:
                    if nnxt not in visited:
                        if self.conflict(nnxt) <= self.conflict(node) + 2:
                            has_valid_forward = True
                            break
                if has_valid_forward or nxt == TRANG_THAI_DICH_PHANG:
                    valid_neighbors.append((nxt, act))
            
            valid_neighbors.sort(key=lambda x: self.conflict(x[0]))
            for nxt, act in valid_neighbors:
                res = backtrack_fc(nxt, path + [act], depth_limit)
                if res is not None:
                    return res
            visited.remove(node)
            return None
        return backtrack_fc(start, [])

    def ac3_search(self, start):
        visited = set()
        def backtrack_ac3(node, path, depth_limit=20):
            if node == TRANG_THAI_DICH_PHANG:
                return path
            if len(path) >= depth_limit:
                return None
            visited.add(node)
            
            neighbors = [x for x in self._hang_xom(node) if x[0] not in visited]
            queue = deque()
            for nxt, _ in neighbors:
                for nnxt, _ in self._hang_xom(nxt):
                    if nnxt not in visited:
                        queue.append((nxt, nnxt))
            
            inconsistent_nodes = set()
            limit_bound = self.conflict(node) + 4
            
            while queue:
                X, Y = queue.popleft()
                if self.conflict(Y) > limit_bound:
                    all_y_bad = True
                    for ny, _ in self._hang_xom(X):
                        if ny not in visited and self.conflict(ny) <= limit_bound:
                            all_y_bad = False
                            break
                    if all_y_bad:
                        if X not in inconsistent_nodes:
                            inconsistent_nodes.add(X)
                            for nxt, _ in neighbors:
                                if nxt != X: queue.append((nxt, X))
            
            valid_neighbors = [x for x in neighbors if x[0] not in inconsistent_nodes]
            if not valid_neighbors: 
                valid_neighbors = neighbors
            
            valid_neighbors.sort(key=lambda x: self.conflict(x[0]))
            for nxt, act in valid_neighbors:
                res = backtrack_ac3(nxt, path + [act], depth_limit)
                if res is not None:
                    return res
            visited.remove(node)
            return None
        return backtrack_ac3(start, [])

    def chay_loi_giai(self, paths):
        self.dang_tu_dong = True
        def step(i):
            if not self.dang_tu_dong: return
            if i >= len(paths):
                self.dang_tu_dong = False
                self._after_id = None
                self.ve_giao_dien()
                return
            self.di_chuyen_o_trong(paths[i])
            self.so_buoc += 1
            self.ve_giao_dien()
            self._after_id = self.cua_so.after(200, lambda: step(i + 1))
        step(0)

    def dung_tu_dong(self):
        self.dang_tu_dong = False
        if self._after_id:
            try: self.cua_so.after_cancel(self._after_id)
            except Exception: pass
            self._after_id = None

    def giai_thuat_toan(self):
        if self.dang_tu_dong: return
        start = tuple(self.ban_co)
        algo = self.bien_thuat_toan.get()
        self.nhan_trang_thai.config(text=f"{algo} đang chạy...", fg="#db7093")
        self.xoa_lich_su()
        self.cua_so.update_idletasks()

        path = None
        if algo == "BFS": path = self.bfs(start)
        elif algo == "DFS": path = self.dfs(start)
        elif algo == "UCS": path = self.ucs(start)
        elif algo == "IDS": path = self.ids(start)
        elif algo == "A* Search": path = self.astar(start)
        elif algo == "IDA*": path = self.ida_star(start)
        elif algo == "Greedy": path = self.greedy(start)
        elif algo == "Simple Hill Climbing": path = self.simple_hill(start)
        elif algo == "Steepest Ascent Hill": path = self.steepest_ascent(start)
        elif algo == "Stochastic Hill": path = self.stochastic_hill(start)
        elif algo == "Random Restart Hill": path = self.random_restart_hill(start)
        elif algo == "Simulated Annealing": path = self.simulated_annealing(start)
        elif algo == "Local Beam": path = self.local_beam(start)
        elif algo == "Belief Search": path = self.belief_search(start)
        elif algo == "Belief Goal": path = self.belief_goal(start)
        elif algo == "Backtracking": path = self.backtracking(start)
        elif algo == "And-Or Search": path = self.and_or_search(start)
        elif algo == "Forward Checking": path = self.forward_checking_search(start)
        elif algo == "AC3": path = self.ac3_search(start)

        if path is None:
            if "Hill" in algo or "Beam" in algo:
                self.nhan_trang_thai.config(text=f"Thất bại: {algo} bị kẹt ở cực trị.", fg="red")
            else:
                self.nhan_trang_thai.config(text=f"Thất bại: {algo} không tìm thấy lời giải.", fg="red")
            return

        self.nhan_trang_thai.config(text=f"{algo} thành công: {len(path)} bước.", fg="#004d80")
        self.xuat_lich_su_ra_man_hinh(path)
        self.chay_loi_giai(path)

if __name__ == "__main__":
    cua_so_chinh = tk.Tk()
    GiaoDienTongHopTatCa(cua_so_chinh)
    cua_so_chinh.mainloop()